# Model Training

## Baseline Model on Initial processed data

In [2]:
import pandas as pd
import numpy as np

In [3]:
data=pd.read_csv(r'C:\ABHIRAM\projects\FifaWCPredictor\data\Processed_result.csv')

In [4]:
data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tier,outcome_home,elo_home,elo_away,elo_diff
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,4,1.0,1500.0,1500.0,0.0
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,4,1.0,1500.0,1495.0,5.0
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,4,0.5,1500.0,1500.0,0.0
3,2000-01-09,Mexico,Iran,2.0,1.0,Friendly,Oakland,United States,True,4,1.0,1500.0,1500.0,0.0
4,2000-01-09,Ivory Coast,Egypt,2.0,0.0,Friendly,Abidjan,Ivory Coast,False,4,1.0,1500.0,1505.0,-5.0


In [5]:
data['date'].dtype

<StringDtype(storage='python', na_value=nan)>

In [6]:
data['date']=pd.to_datetime(data['date'])

In [7]:
data['date'].dtype

dtype('<M8[us]')

In [8]:
data.dtypes

date            datetime64[us]
home_team                  str
away_team                  str
home_score             float64
away_score             float64
tournament                 str
city                       str
country                    str
neutral                   bool
tier                     int64
outcome_home           float64
elo_home               float64
elo_away               float64
elo_diff               float64
dtype: object

In [9]:
conditions=[
    data['outcome_home']==1,
    data['outcome_home']==0.5,
    data['outcome_home']==0
]
values=['home_win','draw','away_win']

data['result']=np.select(conditions,values,default='unknown')

In [10]:
data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tier,outcome_home,elo_home,elo_away,elo_diff,result
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,4,1.0,1500.0,1500.0,0.0,home_win
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,4,1.0,1500.0,1495.0,5.0,home_win
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,4,0.5,1500.0,1500.0,0.0,draw
3,2000-01-09,Mexico,Iran,2.0,1.0,Friendly,Oakland,United States,True,4,1.0,1500.0,1500.0,0.0,home_win
4,2000-01-09,Ivory Coast,Egypt,2.0,0.0,Friendly,Abidjan,Ivory Coast,False,4,1.0,1500.0,1505.0,-5.0,home_win


In [11]:
#baseline model features set

features=['elo_home','elo_away','elo_diff','neutral','tier']
X=data[features]
y=data['result']

In [15]:
train_mask=data['date']<'2023-01-01'
test_mask=data['date']>='2023-01-01'

X_train=X.loc[train_mask]
X_test=X.loc[test_mask]

y_train=y.loc[train_mask]
y_test=y.loc[test_mask]

In [30]:
print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(21745, 5) (3287, 5)
result
home_win    10502
away_win     6167
draw         5076
Name: count, dtype: int64
result
home_win    1546
away_win     991
draw         750
Name: count, dtype: int64


In [19]:
X_train.tail()

,elo_home,elo_away,elo_diff,neutral,tier
21740,1206.519725,1082.398370,124.121355,False,3
21741,1657.145268,1504.993701,152.151567,False,4
21742,1379.142011,1592.240516,-213.098505,False,3
21743,1675.730866,1498.833976,176.896890,False,4
21744,1693.446288,1568.583513,124.862776,True,4


In [20]:
X_test.head()

,elo_home,elo_away,elo_diff,neutral,tier
21745,1580.071086,1218.928595,361.142491,False,3
21746,1365.979521,1473.504305,-107.524784,False,3
21747,1586.775729,1203.091830,383.683899,False,3
21748,1463.721371,1384.606798,79.114573,False,3
21749,1678.384452,1696.722932,-18.338479,False,3


In [29]:
#train logistic regression model
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

model_pipeline=make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000,class_weight='balanced')
)

model_pipeline.fit(X_train,y_train)
y_pred=model_pipeline.predict(X_test)

print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))



0.5765135381807119
[[ 671  135  185]
 [ 306  136  308]
 [ 259  199 1088]]
              precision    recall  f1-score   support

    away_win       0.54      0.68      0.60       991
        draw       0.29      0.18      0.22       750
    home_win       0.69      0.70      0.70      1546

    accuracy                           0.58      3287
   macro avg       0.51      0.52      0.51      3287
weighted avg       0.55      0.58      0.56      3287



BASELINE — Logistic Regression

Features: elo_home, elo_away, elo_diff, neutral, tier

Train: pre-2023, Test: 2023+

Accuracy: 57.65%

Recall — home_win: 0.70, away_win: 0.68, draw: 0.18

Naive majority baseline: ~47%

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

model=RandomForestClassifier()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)


print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))


0.5591724977182841
[[ 547  143  301]
 [ 239  114  397]
 [ 215  154 1177]]
              precision    recall  f1-score   support

    away_win       0.55      0.55      0.55       991
        draw       0.28      0.15      0.20       750
    home_win       0.63      0.76      0.69      1546

    accuracy                           0.56      3287
   macro avg       0.48      0.49      0.48      3287
weighted avg       0.52      0.56      0.53      3287



BASELINE — Random Forest

Features: elo_home, elo_away, elo_diff, neutral, tier

Train: pre-2023, Test: 2023+

Accuracy: 55.92%

Recall — home_win: 0.76, away_win: 0.55, draw: 0.15

Naive majority baseline: ~47%

In [32]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

model=GradientBoostingClassifier()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)


print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))


0.6035898996045026
[[ 632   10  349]
 [ 267    9  474]
 [ 194    9 1343]]
              precision    recall  f1-score   support

    away_win       0.58      0.64      0.61       991
        draw       0.32      0.01      0.02       750
    home_win       0.62      0.87      0.72      1546

    accuracy                           0.60      3287
   macro avg       0.51      0.51      0.45      3287
weighted avg       0.54      0.60      0.53      3287



BASELINE — Gradient Boosting

Features: elo_home, elo_away, elo_diff, neutral, tier

Train: pre-2023, Test: 2023+

Accuracy: 60.35%

Recall — home_win: 0.87, away_win: 0.64, draw: 0.01

Naive majority baseline: ~47%

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1_macro',  # not accuracy!
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)


best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 200}
0.5953757225433526
[[ 615   30  346]
 [ 269   26  455]
 [ 198   32 1316]]
              precision    recall  f1-score   support

    away_win       0.57      0.62      0.59       991
        draw       0.30      0.03      0.06       750
    home_win       0.62      0.85      0.72      1546

    accuracy                           0.60      3287
   macro avg       0.50      0.50      0.46      3287
weighted avg       0.53      0.60      0.53      3287



BASELINE — Gradient Boosting(with tuning)

Features: elo_home, elo_away, elo_diff, neutral, tier

Train: pre-2023, Test: 2023+

Accuracy: 59.53%

Recall — home_win: 0.85, away_win: 0.62, draw: 0.03

Naive majority baseline: ~47%

Model Comparison Table

Model |                          Accuracy |      Draw Recall |     Macro F1

Logistic Regression |            57.65% |        0.18 |            0.51

RandomForest |                   55.91% |        0.15 |            0.48

Gradient Boosting (default) |    60.36% |         0.01 |          0.45

Gradient Boosting (tuned) |      59.54% |         0.03 |           0.46

## Model Training on New featured dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [2]:
data=pd.read_csv(r'C:\ABHIRAM\projects\FifaWCPredictor\data\processed_result_v2.csv')

In [3]:
data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tier,...,elo_diff,form_home,form_away,form_diff,h2h_home,home_avg_goals,home_avg_conceded,away_avg_goals,away_avg_conceded,result
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,4,...,0.0,0.5,0.5,0.0,0.5,1.500000,1.500000,1.500000,1.500000,home_win
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,4,...,5.0,0.5,0.0,0.5,0.5,1.500000,1.500000,1.000000,2.000000,home_win
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,4,...,0.0,0.5,0.5,0.0,0.5,2.500000,2.500000,2.500000,2.500000,draw
3,2000-01-09,Mexico,Iran,2.0,1.0,Friendly,Oakland,United States,True,4,...,0.0,0.5,0.5,0.0,0.5,1.666667,1.666667,1.666667,1.666667,home_win
4,2000-01-09,Ivory Coast,Egypt,2.0,0.0,Friendly,Abidjan,Ivory Coast,False,4,...,-5.0,0.5,1.0,-0.5,0.5,1.625000,1.625000,2.000000,1.000000,home_win


In [4]:
features=[
'neutral',
'elo_home',
'elo_away',
'form_home',
'form_away',
'h2h_home',
'home_avg_goals',
'home_avg_conceded',
'away_avg_goals',
'away_avg_conceded'
]

X=data[features]

In [5]:
conditions=[
    data['outcome_home']==1,
    data['outcome_home']==0.5,
    data['outcome_home']==0
]
values=['home_win','draw','away_win']
data['result']=np.select(conditions,values,default='unknown')
y=data['result']

In [6]:
train_mask=data['date']<'2023-01-01'
test_mask=data['date']>='2023-01-01'

X_train=X.loc[train_mask]
X_test=X.loc[test_mask]

y_train=y.loc[train_mask]
y_test=y.loc[test_mask]

In [7]:
#train logistic regression model
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

model_pipeline=make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000,class_weight='balanced')
)

model_pipeline.fit(X_train,y_train)
y_pred=model_pipeline.predict(X_test)

print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))



0.5783389108609674
[[ 659  160  172]
 [ 277  181  292]
 [ 239  246 1061]]
              precision    recall  f1-score   support

    away_win       0.56      0.66      0.61       991
        draw       0.31      0.24      0.27       750
    home_win       0.70      0.69      0.69      1546

    accuracy                           0.58      3287
   macro avg       0.52      0.53      0.52      3287
weighted avg       0.57      0.58      0.57      3287



BASELINE — Logistic Regression

Features: elo_home, elo_away, elo_diff, neutral, tier, form_home, form_away, form_diff, h2h_home, home_avg_goals, home_avg_conceded, away_avg_goals, away_avg_conceded

Train: pre-2023, Test: 2023+

Accuracy: 58.19%

Recall — home_win: 0.69, away_win: 0.68, draw: 0.23

Naive majority baseline: ~47%

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

model=RandomForestClassifier()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)


print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))


0.5996349254639489
[[ 614   53  324]
 [ 261   58  431]
 [ 194   53 1299]]
              precision    recall  f1-score   support

    away_win       0.57      0.62      0.60       991
        draw       0.35      0.08      0.13       750
    home_win       0.63      0.84      0.72      1546

    accuracy                           0.60      3287
   macro avg       0.52      0.51      0.48      3287
weighted avg       0.55      0.60      0.55      3287



BASELINE — Random Forest

Features: elo_home, elo_away, elo_diff, neutral, tier, form_home, form_away, form_diff, h2h_home, home_avg_goals, home_avg_conceded, away_avg_goals, away_avg_conceded

Train: pre-2023, Test: 2023+

Accuracy: 59.96%

Recall — home_win: 0.84, away_win: 0.62, draw: 0.08

Naive majority baseline: ~47%

In [22]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

model=GradientBoostingClassifier()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)


print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))


0.6029814420444174
[[ 645    5  341]
 [ 273    2  475]
 [ 198   13 1335]]
              precision    recall  f1-score   support

    away_win       0.58      0.65      0.61       991
        draw       0.10      0.00      0.01       750
    home_win       0.62      0.86      0.72      1546

    accuracy                           0.60      3287
   macro avg       0.43      0.51      0.45      3287
weighted avg       0.49      0.60      0.53      3287



BASELINE — Gradient Boosting

Features: elo_home, elo_away, elo_diff, neutral, tier, form_home, form_away, form_diff, h2h_home, home_avg_goals, home_avg_conceded, away_avg_goals, away_avg_conceded

Train: pre-2023, Test: 2023+

Accuracy: 50.29%

Recall — home_win: 0.86, away_win: 0.65, draw: 0.00

Naive majority baseline: ~47%

In [23]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1_macro',  # not accuracy!
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)


best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 200}
0.5932461210830544
[[ 619   40  332]
 [ 258   30  462]
 [ 197   48 1301]]
              precision    recall  f1-score   support

    away_win       0.58      0.62      0.60       991
        draw       0.25      0.04      0.07       750
    home_win       0.62      0.84      0.71      1546

    accuracy                           0.59      3287
   macro avg       0.48      0.50      0.46      3287
weighted avg       0.52      0.59      0.53      3287



BASELINE — Gradient Boosting(with tuning)

Features: elo_home, elo_away, elo_diff, neutral, tier

Train: pre-2023, Test: 2023+

Accuracy: 59.57%

Recall — home_win: 0.86, away_win: 0.61, draw: 0.05

Naive majority baseline: ~47%